# Week 6 — Data Cleaning: Handling Nulls & Data Types
## Phase 2a Python | PORA Academy Cohort 7

By the end of this session, you will be able to:
- Identify and count null values
- Use fillna(), dropna(), and conditional replacement
- Convert data types correctly
- Recognise the typo in the products dataset

---
**Session timing:** 2 hours | DeepSeek assisted ✓

## Why Data Cleaning Matters — The Olist Reality

When Olist sellers list a product, they must fill in a category. But sometimes they skip it. In the real `olist_products_dataset.csv`, **610 out of 32,951 products have no category at all** — they have a null value where a label should be.

That is a problem. If you tried to count sales by category, those 610 products would disappear from your results entirely, making your totals wrong and your decisions based on incomplete data. 

Today you will learn the professional toolkit for dealing with this kind of real-world messiness: how to find missing values, how to decide whether to fill them or remove them, and how to fix dates that are stored as text. By the end of the session, you will be able to take a dataset from "raw and broken" to "analysis-ready" — a skill every data analyst uses every single day.

## Setup — Load the Olist Datasets

Run this cell first. It mounts Google Drive and loads all the datasets we will use today.

In [ ]:
import pandas as pd
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Path to your Olist data folder on Drive
olist_path = '/content/drive/MyDrive/olist-data'

# Load the datasets for today's session
products = pd.read_csv(os.path.join(olist_path, 'olist_products_dataset.csv'))
orders   = pd.read_csv(os.path.join(olist_path, 'olist_orders_dataset.csv'))

print(f"✓ Loaded {len(products):,} products")   # expected: 32,951
print(f"✓ Loaded {len(orders):,} orders")        # expected: 99,441

## Concept 1 — The Products Dataset: A Real Cleaning Challenge

The Olist products dataset is a perfect first cleaning task because it contains **two different kinds of problem at the same time**: missing values AND typos in the column names themselves.

Think of it like receiving a paper form that someone filled in with messy handwriting. Before you can use the information, you have to read through the form carefully, work out what each field is actually supposed to say, and cross out the messy bits and rewrite them cleanly. Only then can the data be trusted.

When you encounter a real-world dataset for the first time, the very first move is always to run `.isnull().sum()` — it tells you exactly which columns have gaps and how many. This gives you a map of the work that lies ahead before you write a single line of analysis code.

In [ ]:
# Step 1 — Check the shape and null counts
print("Shape:", products.shape)              # expected: (32951, 9)
print()
print("Null counts per column:")
print(products.isnull().sum())
# expected output:
# product_id                        0
# product_category_name           610  ← missing categories
# product_name_lenght             610  ← TYPO: should be 'length'
# product_description_lenght      610  ← TYPO: same mistake
# product_photos_qty              610
# product_weight_g                  2
# product_length_cm                 2
# product_height_cm                 2
# product_width_cm                  2

In [ ]:
# Step 2 — Fix the column name typos
# The dataset was created with a Portuguese typo: 'lenght' instead of 'length'
# We must rename them before any downstream code accidentally uses the wrong name

products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

print("Corrected columns:")
print(products.columns.tolist())
# expected: ['product_id', 'product_category_name', 'product_name_length',
#            'product_description_length', 'product_photos_qty',
#            'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

## Concept 2 — fillna & dropna: Choosing Your Null Strategy

When you find a null value, you have two main choices: fill it with something reasonable, or remove the row entirely. The right choice depends on what the null *means* in context.

Imagine you run a school canteen and you have a sign-up list where some students left the "meal preference" column blank. Option one: write "standard meal" in every blank cell so nobody goes hungry — that is `fillna`. Option two: remove those students from the list entirely so you do not cook for uncertain quantities — that is `dropna`. The method you choose changes your data, so you must choose deliberately.

In pandas, `fillna(value)` replaces every null in a column with the value you specify, while `dropna(subset=['column'])` removes any row where that column is null. There is a third variant — `dropna()` with no arguments — which removes a row if **any** column in that row is null, which is often too aggressive.

In [ ]:
# Strategy A — fillna: keep all 32,951 rows, replace nulls with 'unknown'
products_filled = products.copy()
products_filled['product_category_name'] = products_filled['product_category_name'].fillna('unknown')

print("Nulls after fillna:", products_filled['product_category_name'].isnull().sum())  # expected: 0
print("Rows with 'unknown':", (products_filled['product_category_name'] == 'unknown').sum())  # expected: 610

In [ ]:
# Strategy B — dropna on a specific column: remove the 610 uncategorised rows
products_dropped = products.dropna(subset=['product_category_name'])
print("Shape after dropna (by category):", products_dropped.shape)  # expected: (32341, 9)

# Strategy C — dropna on ALL columns: remove rows with ANY null
products_all_clean = products.dropna()
print("Shape after dropna (all nulls):", products_all_clean.shape)  # expected: (32341, 9)

# ── Which strategy to use? ──────────────────────────────────────────────────
# fillna('unknown') → keep the row, the product still exists, just uncategorised
# dropna(subset=...)→ remove incomplete rows that would skew category analysis
# dropna()          → use only when EVERY column must have a value (strict analysis)

## Concept 3 — Datetime Conversion & Delivery Time Calculation

When pandas reads a CSV file, it reads everything as text. A value like `"2018-03-15 12:45:30"` looks like a date to you but it is just a string to pandas — you cannot add or subtract strings the way you can with dates.

`pd.to_datetime()` converts those strings into real datetime objects that pandas understands as points in time. Once a column is datetime, you can subtract two columns to get a duration (like delivery time), extract the year or month, and sort chronologically.

The magic ingredient is `errors='coerce'`: if a value cannot be parsed as a date (e.g., it is blank or contains garbage text), pandas replaces it with `NaT` (Not a Time) instead of throwing an error and stopping. This is the safe, professional approach — it lets you convert an entire column in one step without being derailed by a handful of bad values.

In [ ]:
# Convert all five date columns from strings to proper datetimes
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Confirm the conversion worked — dtype should now be datetime64
print("Data types after conversion:")
print(orders[date_cols].dtypes)
# expected: all columns show datetime64[ns]

In [ ]:
# Calculate delivery time in days
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

# Filter to delivered orders only, then get descriptive statistics
delivered = orders[orders['order_status'] == 'delivered']

print(f"Delivered orders: {len(delivered):,}")  # expected: 96,470
print()
print("Delivery time stats (days):")
print(delivered['delivery_days'].describe().round(1))
# expected:
# count    96470.0
# mean        12.1
# std          9.6
# min          0.0
# 25%          6.0
# 50%         10.0
# 75%         15.0
# max        209.0

## Using DeepSeek for Data Cleaning Tasks

From Week 4 you have been using DeepSeek as a coding partner. Today's session is a great context for that — data cleaning tasks often involve patterns that are worth getting AI help on.

**The 4 rules when using DeepSeek:**

1. **Understand first** — before prompting, describe in plain English what you want to achieve
2. **Prompt with context** — give DeepSeek the DataFrame name, relevant column names, and the specific question
3. **Validate the output** — every output from DeepSeek must be checked against expected values from the curriculum
4. **Never copy blindly** — you must be able to explain every line of generated code

**Example prompt for today:**

> *"I have a pandas DataFrame called `orders` with a column `order_purchase_timestamp` that is stored as strings. I want to convert it to datetime and then calculate the number of days between purchase and delivery. The delivered_customer_date column is called `order_delivered_customer_date`. How do I do this?"*

The curriculum tells you the expected answer: mean delivery ≈ 12.1 days, max = 209 days. If DeepSeek's code produces different numbers, something is wrong — fix the code, not the expected value.

## §4 — Going Deeper: Chained Cleaning in One Step

Once you understand `fillna` and `dropna` separately, you can chain them with other pandas operations in a single, readable pipeline. This is the style you will see in production data science code.

A common gotcha: when you use `.dropna()` and then reset the index, the original row numbers are preserved by default. This causes confusing jumps in the index (e.g., row 0, row 1, row 4 — skipping 2 and 3). Always use `reset_index(drop=True)` after `dropna()` if you want a clean 0, 1, 2 … sequence.

In [ ]:
# Chained cleaning — fix typos, fill nulls, select useful columns, reset index
products_clean = (
    products
    .rename(columns={
        'product_name_lenght': 'product_name_length',
        'product_description_lenght': 'product_description_length'
    })
    .fillna({'product_category_name': 'unknown'})
    [['product_id', 'product_category_name', 'product_weight_g']]
    .reset_index(drop=True)
)

print("Cleaned shape:", products_clean.shape)       # expected: (32951, 3)
print("Remaining nulls:", products_clean.isnull().sum().sum())  # expected: 2 (weight_g)
print()
print(products_clean.head(3))

## §5 — Common Mistakes

Two mistakes appear repeatedly when students first work with nulls and datetimes.

In [ ]:
# ── COMMON MISTAKE 1: Checking nulls with == None ──────────────────────
# WRONG — this never finds any null rows in pandas:
# null_rows = products[products['product_category_name'] == None]
# (== None always returns False in pandas, even for actual nulls)

# CORRECT — use .isna() or .isnull():
null_rows = products[products['product_category_name'].isna()]
print("Rows with null category:", len(null_rows))  # expected: 610

# ── COMMON MISTAKE 2: Forgetting errors='coerce' ───────────────────────────
# WRONG — if any value cannot be parsed, this raises a ValueError and stops:
# orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])

# CORRECT — errors='coerce' turns bad values into NaT instead of crashing:
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'], errors='coerce')
nat_count = orders['order_approved_at'].isna().sum()
print(f"NaT values in order_approved_at: {nat_count}")  # expected: small number (orders not yet approved)

## §6 — Mini-Challenge

You are a data analyst at Olist investigating **heavy products**. A product is "heavy" if its `product_weight_g` is greater than 10,000 grams (10 kg).

**Your task:**
1. Start from `products_filled` (the version with 'unknown' category already applied)
2. Drop rows where `product_weight_g` is null
3. Count how many products weigh more than 10,000 g
4. Print the result

**Expected output:** a number between 1,000 and 3,000 heavy products.

⏱ ~5 minutes

In [ ]:
# Mini-Challenge — given variables (do not change)
products_filled = products.copy()
products_filled['product_category_name'] = products_filled['product_category_name'].fillna('unknown')

# Your code here — drop rows where product_weight_g is null
# products_with_weight = products_filled.dropna(subset=['product_weight_g'])

# Your code here — count products heavier than 10,000 g
# heavy_count = (products_with_weight['product_weight_g'] > 10000).sum()

# print(f'Products heavier than 10 kg: {heavy_count:,}')

## §7 — Group Exercise (20 minutes)

Work in your groups. Each task builds on the previous one — verify each expected output before moving to the next.

**Task 1:** Load the products dataset. Print the null counts. Confirm you see 610 nulls in `product_category_name`.  
**Task 2:** Fix the two column name typos (`product_name_lenght` → `product_name_length`, same for description). Verify the corrected column names appear in `.columns.tolist()`.  
**Task 3:** Fill null `product_category_name` with `'unknown'`. Verify: 0 nulls remain and exactly 610 rows contain `'unknown'`.  
**Task 4:** Load `olist_orders_dataset.csv`. Convert all 5 date columns using `pd.to_datetime(errors='coerce')`. Calculate `delivery_days`. Verify: mean ≈ 12.1 days.  
**Task 5 (stretch):** What percentage of delivered orders took more than 30 days?  
*Hint: `delivered['delivery_days'].gt(30).mean() * 100`*

In [ ]:
# Group Exercise — all input data is provided; fill in the blanks

# Task 1 — load and inspect products
products_raw = pd.read_csv(os.path.join(olist_path, 'olist_products_dataset.csv'))
print(products_raw.shape)           # expected: (32951, 9)
print(products_raw.isnull().sum())  # expected: 610 in category and name/desc columns

# Task 2 — fix column name typos
products_clean = products_raw.rename(columns={
    # Your code here
})
print(products_clean.columns.tolist())

# Task 3 — fill nulls in product_category_name
# Your code here

# Task 4 — load orders, convert dates, calculate delivery_days
orders_raw = pd.read_csv(os.path.join(olist_path, 'olist_orders_dataset.csv'))
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
# Your code here (convert date columns, calculate delivery_days)

# Task 5 (stretch) — % of delivered orders taking >30 days
# Your code here

## §8 — Session Summary

| Concept | Key Syntax | Example |
|---|---|---|
| Count nulls | `df['col'].isnull().sum()` | `610` missing categories |
| Fill nulls | `df['col'].fillna('unknown')` | Replace blanks with `'unknown'` |
| Drop null rows | `df.dropna(subset=['col'])` | Remove 610 uncategorised rows |
| Fix column names | `df.rename(columns={'old': 'new'})` | Fix `lenght` → `length` typo |
| Convert to datetime | `pd.to_datetime(col, errors='coerce')` | Parse all 5 date columns safely |
| Calculate duration | `(date2 - date1).dt.days` | Delivery time in integer days |

---

**Coming up Thursday**: String Cleaning in Pandas — using the `.str` accessor to clean city names, search text, and standardise category labels. You will also use DeepSeek to tackle a multi-step city ranking challenge.